# Hamiltonian Neural Networks — Results

This notebook trains HNN and baseline MLP models on three physical systems and visualizes the key results:

1. **Phase space trajectories** — true vs HNN vs baseline (do the orbits close?)
2. **Energy conservation** — H(t) over long rollouts (the money plot)
3. **Learned Hamiltonian contours** — does the network learn the right energy landscape?

Based on: Greydanus, Dzamba & Sosanya, "Hamiltonian Neural Networks" (NeurIPS 2019)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

from src.systems import SYSTEMS, generate_trajectories
from src.hnn import HNN
from src.baseline import BaselineMLP
from src.train import train_system
from src.evaluate import rollout, true_rollout, compute_energy

## 1. Train Models

Train HNN and baseline on all three systems (2000 epochs each).

In [ ]:
results = {}
for system_name in ['spring', 'pendulum', 'twobody']:
    results[system_name] = train_system(system_name, epochs=2000, save_dir='../models')

### Training Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, name in enumerate(['spring', 'pendulum', 'twobody']):
    ax = axes[i]
    ax.semilogy(results[name]['hnn_history']['test_loss'], label='HNN')
    ax.semilogy(results[name]['baseline_history']['test_loss'], label='Baseline')
    ax.set_title(name.capitalize())
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Test Loss (MSE)')
    ax.legend()
    ax.grid(True, alpha=0.3)
fig.suptitle('Training Loss Curves', fontsize=14)
fig.tight_layout()
plt.show()

## 2. Long-Rollout Comparison

Roll out trajectories for 150+ time units (100+ oscillation periods for the spring). This is where the HNN's energy conservation advantage becomes clear.

In [ ]:
def run_rollouts(system_name, models, n_steps=5000, dt=0.02):
    """Run rollouts for HNN, baseline, and ground truth."""
    sys_info = SYSTEMS[system_name]
    rng = np.random.default_rng(123)
    x0_np = sys_info['initial_conditions'](1, rng=rng)[0]
    x0 = torch.tensor(x0_np, dtype=torch.float32)
    
    true_traj = true_rollout(system_name, x0_np, dt, n_steps)
    hnn_traj = rollout(models['hnn'], x0, dt, n_steps)
    bl_traj = rollout(models['baseline'], x0, dt, n_steps)
    t = np.arange(n_steps + 1) * dt
    
    return {'true': true_traj, 'hnn': hnn_traj, 'baseline': bl_traj, 't': t}

rollouts = {}
for name in ['spring', 'pendulum', 'twobody']:
    rollouts[name] = run_rollouts(name, results[name])
    print(f"{name}: done")

### Phase Space Trajectories

The HNN orbits should close (conserving energy). The baseline should spiral outward or inward (energy drift).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, name in enumerate(['spring', 'pendulum', 'twobody']):
    ax = axes[i]
    r = rollouts[name]
    dof = SYSTEMS[name]['dof']
    
    if dof == 1:
        ax.plot(r['true'][:, 0], r['true'][:, 1], 'k-', lw=2, alpha=0.7, label='Ground truth')
        ax.plot(r['hnn'][:, 0], r['hnn'][:, 1], 'b-', lw=1, alpha=0.8, label='HNN')
        ax.plot(r['baseline'][:, 0], r['baseline'][:, 1], 'r-', lw=1, alpha=0.8, label='Baseline')
        ax.set_xlabel('q (position)')
        ax.set_ylabel('p (momentum)')
    else:
        ax.plot(r['true'][:, 0], r['true'][:, 1], 'k-', lw=2, alpha=0.7, label='Ground truth')
        ax.plot(r['hnn'][:, 0], r['hnn'][:, 1], 'b-', lw=1, alpha=0.8, label='HNN')
        ax.plot(r['baseline'][:, 0], r['baseline'][:, 1], 'r-', lw=1, alpha=0.8, label='Baseline')
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_aspect('equal')
    
    ax.set_title(name.capitalize(), fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

fig.suptitle('Phase Space Trajectories (T=100, dt=0.02)', fontsize=16)
fig.tight_layout()
plt.show()

### Energy Conservation (The Money Plot)

Total energy H(t) over long rollouts. The HNN line should stay flat; the baseline should drift.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(['spring', 'pendulum', 'twobody']):
    ax = axes[i]
    r = rollouts[name]
    
    E_true = compute_energy(r['true'], name)
    E_hnn = compute_energy(r['hnn'], name)
    E_bl = compute_energy(r['baseline'], name)
    
    ax.plot(r['t'], E_true, 'k-', lw=2, label='Ground truth')
    ax.plot(r['t'], E_hnn, 'b-', lw=1.5, label='HNN')
    ax.plot(r['t'], E_bl, 'r-', lw=1.5, label='Baseline')
    ax.set_xlabel('Time')
    ax.set_ylabel('Total Energy H')
    ax.set_title(name.capitalize(), fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Print drift stats
    hnn_drift = np.abs(E_hnn - E_true[0]).max()
    bl_drift = np.abs(E_bl - E_true[0]).max()
    print(f"{name}: HNN max drift = {hnn_drift:.6f}, Baseline max drift = {bl_drift:.6f}")

fig.suptitle('Energy Conservation Over Time', fontsize=16)
fig.tight_layout()
plt.show()

### Learned vs True Hamiltonian Contours

For the 1D systems (spring & pendulum), compare the energy landscape learned by the HNN with the true Hamiltonian. The contour shapes should match (up to a global constant/offset, since H is only defined up to a constant).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for row, name in enumerate(['spring', 'pendulum']):
    sys = SYSTEMS[name]
    hnn = results[name]['hnn']
    
    if name == 'spring':
        q_range = np.linspace(-2.5, 2.5, 100)
        p_range = np.linspace(-2.5, 2.5, 100)
    else:
        q_range = np.linspace(-3.5, 3.5, 100)
        p_range = np.linspace(-10, 10, 100)
    
    Q, P = np.meshgrid(q_range, p_range)
    grid = np.column_stack([Q.ravel(), P.ravel()])
    
    # True H
    H_true = sys['hamiltonian'](grid[:, 0:1], grid[:, 1:2], **sys['params']).reshape(Q.shape)
    
    # Learned H
    hnn.eval()
    with torch.no_grad():
        H_learned = hnn.hamiltonian(torch.tensor(grid, dtype=torch.float32)).numpy().reshape(Q.shape)
    
    levels = 20
    axes[row, 0].contour(Q, P, H_true, levels=levels, cmap='viridis')
    axes[row, 0].set_title(f'{name.capitalize()} — True H', fontsize=13)
    axes[row, 0].set_xlabel('q')
    axes[row, 0].set_ylabel('p')
    
    axes[row, 1].contour(Q, P, H_learned, levels=levels, cmap='viridis')
    axes[row, 1].set_title(f'{name.capitalize()} — Learned H (HNN)', fontsize=13)
    axes[row, 1].set_xlabel('q')
    axes[row, 1].set_ylabel('p')
    
    for ax in axes[row]:
        ax.set_aspect('equal' if name == 'spring' else 'auto')
        ax.grid(True, alpha=0.3)

fig.suptitle('Hamiltonian Contours: True vs Learned', fontsize=16)
fig.tight_layout()
plt.show()

## Summary

The Hamiltonian Neural Network learns dynamics that **automatically conserve energy** because the architecture enforces Hamilton's equations via autograd. The baseline MLP, despite similar training loss, accumulates energy drift during long rollouts — orbits spiral instead of closing.

Key takeaway: encoding physical structure (symplectic geometry) into the network architecture is far more powerful than hoping the network learns conservation laws from data alone.